In [ ]:
import pandas as pd
import numpy as np

df = pd.read_excel("../data/Telco_Customer_Churn.xlsx")

In [ ]:
# Lower each column name
df.columns = df.columns.str.lower()

# Drop irrelevant information
df = df.drop(columns=["customerid", "count", "lat long", "latitude", "longitude", "churn label"], errors = "ignore")

In [ ]:
print(f"Amount of total charges that are null: {df["total charges"].isnull().sum()}")

df["total charges"] = pd.to_numeric(df["total charges"], errors = "coerce") # convert invalid to NaN
df["total charges"] = df["total charges"].fillna(0) # convert NaN to to 0



In [ ]:
columns_to_change_to_binary = ["gender", "senior citizen", "partner", "dependents", "phone service", "multiple lines", "online security", "online backup", "device protection", "tech support", "streaming tv", "streaming movies", "paperless billing"]

for column in columns_to_change_to_binary:
    df[column] = df[column].map({
        "Female":0, 
        "Male":1,
        "No":0,
        "No phone service":0,
        "No internet service":0,
        "Yes":1,
        })



In [ ]:
subscription_services = ["phone service", "multiple lines", "online security", "online backup", "device protection", "tech support", "streaming tv", "streaming movies"]

df["number of subscriptions"] = df[subscription_services].apply(lambda row: (row == 1).sum(), axis=1)

df["average charge per subscription"] = df.apply(lambda row: row["monthly charges"] / row["number of subscriptions"] if row["number of subscriptions"] > 0 else 0,
                                                 axis = 1)

print(f"Maximum number of months a customer has been subscribed: {df["tenure months"].max()}\n")

df["tenure month type"] = pd.cut( # group tenure months into bins
    df["tenure months"],
    bins = [0, 12, 24, 48, 72],
    labels = ["0<=m<12", "12<m<24", "24<m<48", "48<m<=72"]
)

print(df[["number of subscriptions", "average charge per subscription", "tenure month type"]].head(10).to_string(index=False))


In [ ]:
df["contract"] = df["contract"].map({
    "Month-to-month":0,
    "One year":1,
    "Two year":2
    })



In [ ]:
# One hot encoding categorical variables
pd.get_dummies(df, columns=["internet service", "payment method", "tenure month type"]) # bool type

columns_with_bool_type = df.select_dtypes(include="bool").columns
df[columns_with_bool_type] = df[columns_with_bool_type].astype(int) # convert to 0/1

In [ ]:
churn_counts = df["churn value"].value_counts()
churn_percentage = df["churn value"].value_counts(normalize=True) * 100 # percent version

print("Class distribution:")
for label, count, pct in zip(churn_counts.index, churn_counts, churn_percentage): # guaranteed same length
    print(f"{'Churned' if label == 1 else 'Retained'}: {count} ({pct:.1f}%)")